# Exploratory & Reproducibility — Identity Fields, Leakage, and Artifacts

This notebook runs **no models**. It reproduces, directly from the raw data, every factual claim
the paper makes about (i) identity-field availability, (ii) leakage-by-design, and (iii) collection
artifacts.

**Inputs (place under `./data/`):**
- `Dataset_T-ITS.csv` — raw published UAV cyber-physical dataset (Hassler et al., T-ITS 2023).
- `cyber_clean.csv`, `cyber_clean_clean.csv` — our derived cyber-only views.
- `UAVIDS-2025.csv` — simulation flow-level dataset.

**Outputs:** Table 1 (BSSID degeneracy), Table 2 (identity cardinality per class),
IP-layer residual / artifact evidence, range-based leakage demo, data-quality count,
UAVIDS identity-richness contrast.

In [ ]:
# Setup
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
pd.set_option("display.width",140); pd.set_option("display.max_columns",60)
DATA="/content/data/"   # repo-relative; put the CSVs here
RANGES_CYBER={"benign":(1,9426),"dos":(13718,25389),"replay":(26364,38370),
              "evil_twin":(39345,45028),"fdi":(50503,53976)}  # from dataset readme (1-based, inclusive)
def savefig(name):
    plt.tight_layout(); plt.savefig(name,dpi=200,bbox_inches="tight"); plt.close(); print("saved",name)
print("ready")

ready


## 0. Pipeline trace — raw → cyber_clean → cyber_clean_clean

Confirms our derivation is faithful (row counts, column deltas). The cyber row count must equal the
sum of the readme cyber ranges.

In [ ]:
raw=pd.read_csv(DATA+"Dataset_T-ITS.csv", low_memory=False)
cc =pd.read_csv(DATA+"cyber_clean.csv", low_memory=False)
ccc=pd.read_csv(DATA+"cyber_clean_clean.csv", low_memory=False)
exp_cyber=sum(b-a+1 for a,b in RANGES_CYBER.values())
print(f"raw rows               : {len(raw)}")
print(f"expected cyber rows    : {exp_cyber}  (sum of readme cyber ranges)")
print(f"cyber_clean rows       : {len(cc)}")
print(f"cyber_clean_clean rows : {len(ccc)}")
print(f"cols dropped clean->clean_clean: {sorted(set(cc.columns)-set(ccc.columns))}")
assert len(cc)==exp_cyber, "row count mismatch!"
# attach cyber attack label to raw by row number (1-based)
raw=raw.reset_index(drop=True); raw["_rownum"]=np.arange(1,len(raw)+1)
def _lab(n):
    for k,(a,b) in RANGES_CYBER.items():
        if a<=n<=b: return k
    return "physical/gap"
raw["cyber_label"]=raw["_rownum"].map(_lab)
print("\nraw cyber-label counts:\n", raw["cyber_label"].value_counts())

raw rows               : 54783
expected cyber rows    : 42263  (sum of readme cyber ranges)
cyber_clean rows       : 42263
cyber_clean_clean rows : 42263
cols dropped clean->clean_clean: ['class', 'segment']

raw cyber-label counts:
 cyber_label
physical/gap    12520
replay          12007
dos             11672
benign           9426
evil_twin        5684
fdi              3474
Name: count, dtype: int64


## 1. Table 1 — `wlan.bssid` is degenerate in the SOURCE (not from cleaning)

Claim: the published capture records BSSID as a constant `'0'` across **all** classes; our cleaning
preserves it unchanged. Therefore the evil-twin attack's natural MAC/AP-identity signature is absent
from the data — a capture limitation, not a preprocessing artifact.

In [ ]:
def bssid_table(df,labelcol,name):
    rows=[]
    for k in df[labelcol].astype(str).unique():
        b=df[df[labelcol].astype(str)==k]["wlan.bssid"].astype(str)
        top=b.value_counts().head(1)
        rows.append({"source":name,"class":k,"bssid_nunique":int(b.nunique()),
                     "dominant_value":top.index[0],"dominant_share":round(top.iloc[0]/len(b),4)})
    return pd.DataFrame(rows)
T1=pd.concat([bssid_table(raw[raw.cyber_label!="physical/gap"],"cyber_label","raw"),
              bssid_table(cc,"attack_type","cyber_clean"),
              bssid_table(ccc,"attack_type","cyber_clean_clean")],ignore_index=True)
print("=== TABLE 1: wlan.bssid per class across the pipeline ===")
print(T1.to_string(index=False))
T1.to_csv("table1_bssid_degeneracy.csv",index=False); print("\nsaved table1_bssid_degeneracy.csv")

=== TABLE 1: wlan.bssid per class across the pipeline ===
           source     class  bssid_nunique dominant_value  dominant_share
              raw    benign              2              0          0.9999
              raw       dos              2              0          0.9999
              raw    replay              2              0          0.9999
              raw evil_twin              2              0          0.9998
              raw       fdi              2              0          0.9997
      cyber_clean    benign              2              0          0.9999
      cyber_clean       dos              2              0          0.9999
      cyber_clean    replay              2              0          0.9999
      cyber_clean evil_twin              2              0          0.9998
      cyber_clean       fdi              2              0          0.9997
cyber_clean_clean    benign              2              0          0.9999
cyber_clean_clean       dos              2            

## 2. Table 2 — identity-field cardinality per class (T-ITS cyber)

Claim: MAC-layer identity fields carry too little diversity for identity-structure features
(small testbed). The lone exception is `ip.src` for evil_twin (see §3).

In [ ]:
ID_COLS=["wlan.bssid","wlan.sa","wlan.ta","wlan.da","wlan.ra","ip.src","ip.dst"]
def card_table(df,labelcol):
    out={}
    for c in ID_COLS:
        out[c]={k:int(df[df[labelcol].astype(str)==k][c].nunique(dropna=True))
                for k in df[labelcol].astype(str).unique()}
    return pd.DataFrame(out)
T2=card_table(cc,"attack_type")
print("=== TABLE 2: identity-field nunique per class (cyber_clean) ===")
print(T2.to_string())
T2.to_csv("table2_identity_cardinality.csv"); print("\nsaved table2_identity_cardinality.csv")

# figure: heatmap-ish bar of cardinalities
fig,ax=plt.subplots(figsize=(8,3.4))
T2.T.plot(kind="bar",ax=ax,logy=True); ax.set_ylabel("nunique (log)"); ax.set_xlabel("identity field")
ax.set_title("T-ITS: identity-field cardinality per class"); ax.legend(fontsize=7,ncol=5)
savefig("fig_identity_cardinality.png")

=== TABLE 2: identity-field nunique per class (cyber_clean) ===
           wlan.bssid  wlan.sa  wlan.ta  wlan.da  wlan.ra  ip.src  ip.dst
benign              2        3        3       15       15       3       6
dos                 2        2        2        5        5       3       8
replay              2        3        3        7        7       4       8
evil_twin           2        5        2        3        2    2420       2
fdi                 2        9       42        1       42       2       2

saved table2_identity_cardinality.csv
saved fig_identity_cardinality.png


## 3. Evil-twin residual is IP-layer and **artifact-like** (unusable)

`ip.src` shows a huge cardinality spike for evil_twin only. We test whether it is a generalizable
signal or a class-exclusive collection artifact: single-feature separability and class-purity.
High purity + near-perfect single-feature AUC ⇒ artifact (same family as FlowID/row-index leakage).

In [ ]:
sub=cc.copy(); sub["is_evil"]=(sub["attack_type"].astype(str)=="evil_twin").astype(int)
# encode ip.src as integer id (value-as-label, to expose memorization)
ipsrc_id=sub["ip.src"].astype("category").cat.codes.values
auc_ipsrc=roc_auc_score(sub["is_evil"],ipsrc_id); auc_ipsrc=max(auc_ipsrc,1-auc_ipsrc)
# class-purity of ip.src: fraction of ip.src values that are class-exclusive
tab=pd.crosstab(sub["ip.src"].astype(str),sub["attack_type"].astype(str))
share=tab.max(axis=1)/tab.sum(axis=1); excl=float((share==1.0).mean())
print(f"evil_twin ip.src nunique        : {sub.loc[sub.is_evil==1,'ip.src'].nunique()}  "
      f"(other classes max = {sub.loc[sub.is_evil==0].groupby('attack_type')['ip.src'].nunique().max()})")
print(f"ip.src class-exclusive fraction : {excl:.3f}  -> {'ARTIFACT (>=0.99 class-exclusive)' if excl>=0.99 else 'check'}")
print(f"ip.src single-feature AUC (evil): {auc_ipsrc:.3f}  (supporting; category-coded)")
print("Decisive criterion = class-exclusivity: ~all evil_twin ip.src values appear in no other class")
print("=> residual signal is identity memorization, not a deployable/generalizable detector.")

evil_twin ip.src nunique        : 2420  (other classes max = 4)
ip.src class-exclusive fraction : 0.998  -> ARTIFACT (>=0.99 class-exclusive)
ip.src single-feature AUC (evil): 0.983  (supporting; category-coded)
Decisive criterion = class-exclusivity: ~all evil_twin ip.src values appear in no other class
=> residual signal is identity memorization, not a deployable/generalizable detector.


## 4. Leakage-by-design — labels are defined by row ranges

Because the readme assigns classes by contiguous row ranges, **any** ordering/identifier feature
(row index, FlowID, timestamp) reproduces the label almost perfectly. We demonstrate with row index
on T-ITS and with `FlowID` on UAVIDS — both should give artifact-level AUC.

In [ ]:
# T-ITS: row index vs label (multi-class -> one-vs-rest mean AUC)
rownum=raw.loc[raw.cyber_label!="physical/gap","_rownum"].values
ylab=raw.loc[raw.cyber_label!="physical/gap","cyber_label"].values
aucs=[]
for k in np.unique(ylab):
    aucs.append(max(roc_auc_score((ylab==k).astype(int),rownum),1-roc_auc_score((ylab==k).astype(int),rownum)))
print(f"T-ITS row-index one-vs-rest mean AUC = {np.mean(aucs):.3f}  (leakage-by-design)")

# UAVIDS: FlowID vs label (read a FRESH copy; do not mutate a shared frame)
uav_fid=pd.read_csv(DATA+"UAVIDS-2025.csv", low_memory=False)
if "FlowID" in uav_fid.columns:
    fid=uav_fid["FlowID"].astype("category").cat.codes.values
    a=[]
    for k in uav_fid["label"].astype(str).unique():
        s=roc_auc_score((uav_fid["label"].astype(str)==k).astype(int),fid); a.append(max(s,1-s))
    print(f"UAVIDS FlowID one-vs-rest mean AUC   = {np.mean(a):.3f}  -> identifier encodes the label")

T-ITS row-index one-vs-rest mean AUC = 0.859  (leakage-by-design)
UAVIDS FlowID one-vs-rest mean AUC   = 0.724  -> identifier encodes the label


## 5. Data quality — embedded/corrupt rows

A handful of rows contain embedded header/garbage tokens (non-numeric `frame.len`). We report and
filter them; impact is negligible but disclosed for transparency.

In [ ]:
fl=pd.to_numeric(cc["frame.len"],errors="coerce")
bad=cc[fl.isna()]
print(f"cyber_clean corrupt rows (non-numeric frame.len): {len(bad)}")
if len(bad):
    print("example junk tokens in wlan.bssid:", list(bad["wlan.bssid"].astype(str).unique()[:6]))

cyber_clean corrupt rows (non-numeric frame.len): 5
example junk tokens in wlan.bssid: ['distance', 'vgy', 'yaw']


## 6. Contrast — UAVIDS-2025 HAS usable identity structure

Where identity-based detection is *possible*: UAVIDS `SrcAddr` shows real per-source diversity and
is NOT class-exclusive (purity well below 1), so value-agnostic source-rate features are meaningful
rather than memorization. This is why the positive result lives on UAVIDS.

In [ ]:
uav=pd.read_csv(DATA+"UAVIDS-2025.csv", low_memory=False)   # FRESH read, unmutated
print("UAVIDS shape:", uav.shape)
src=uav["SrcAddr"].astype(str)
tab=pd.crosstab(src,uav["label"].astype(str)); share=tab.max(axis=1)/tab.sum(axis=1)
w=tab.sum(axis=1)/tab.values.sum()
print(f"UAVIDS SrcAddr: n_values={src.nunique()}  weighted_purity={float((share*w).sum()):.3f}  "
      f"class_exclusive_frac={float((share==1.0).mean()):.3f}")
print(f"UAVIDS DstAddr: n_values={uav['DstAddr'].astype(str).nunique()}")
# decisive: are Sybil sources SHARED with other classes? (shared => behaviour, not identity memorization)
syb=set(uav.loc[uav['label'].astype(str).str.contains('Sybil'),'SrcAddr'].astype(str))
non=set(uav.loc[~uav['label'].astype(str).str.contains('Sybil'),'SrcAddr'].astype(str))
print(f"Sybil sources={len(syb)}  shared_with_other_classes={len(syb & non)}  "
      f"-> shared => source-RATE features capture behaviour, not identity")
# small comparison figure: identity usability T-ITS vs UAVIDS
fig,ax=plt.subplots(figsize=(5.2,3.2))
labels=["T-ITS\nwlan.bssid","T-ITS\nwlan.sa","UAVIDS\nSrcAddr"]
vals=[cc["wlan.bssid"].nunique(),cc["wlan.sa"].nunique(),src.nunique()]
ax.bar(labels,vals); ax.set_yscale("log"); ax.set_ylabel("distinct identity values (log)")
ax.set_title("Identity diversity: where identity-based detection is feasible")
savefig("fig_identity_usability.png")

UAVIDS shape: (122171, 23)
UAVIDS SrcAddr: n_values=176  weighted_purity=0.588  class_exclusive_frac=0.290
UAVIDS DstAddr: n_values=218
Sybil sources=37  shared_with_other_classes=15  -> shared => source-RATE features capture behaviour, not identity
saved fig_identity_usability.png
